In [5]:
import pandas as pd

# 1. Cargamos las Fuentes de Información:
# Importamos los reportes "raw" (crudos) de dos sistemas independientes: 
# El ERP interno (Libro Mayor) y el extracto externo (Home Banking).
df_mayor = pd.read_csv("../data/raw/libro_mayor.csv")
df_banco = pd.read_csv("../data/raw/extracto_bancario.csv")

# 2. Diagnóstico Inicial:
# Realizamos una inspección visual para validar la estructura de las columnas y que el contenido
# coincida con lo esperado para el proceso de conciliación.
print("--- LIBRO MAYOR (Primeras 3 filas) ---")
display(df_mayor.head(3))

print("\n--- EXTRACTO BANCARIO (Primeras 3 filas) ---")
display(df_banco.head(3))

# 3. Auditoría de Tipos de Datos:
# Verificamos la consistencia técnica. Para un contador, asegurar que los montos 
# sean flotantes y las fechas sean procesables es el primer control de calidad.
print("\n--- INFO DEL LIBRO MAYOR ---")
df_mayor.info()

--- LIBRO MAYOR (Primeras 3 filas) ---


,ref_comprobante,fecha_contable,monto,descripcion
0,TRX-1001,2026-03-07,-79739.55,Pago - Proveedor_2
1,TRX-1002,2026-03-17,92790.60,Cobro - Cliente_15
2,TRX-1003,2026-03-20,-24229.57,Pago - Aguas Cordobesas



--- EXTRACTO BANCARIO (Primeras 3 filas) ---


,ref_comprobante,fecha_banco,monto,descripcion
0,TRX-1001,2026-03-09,79749.55,BCO_MOV_PROV
1,TRX-1002,2026-03-17,-92790.60,BCO_MOV_CLIE
2,TRX-1003,2026-03-20,24229.57,BCO_MOV_AGUA



--- INFO DEL LIBRO MAYOR ---
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ref_comprobante  1000 non-null   str    
 1   fecha_contable   1000 non-null   str    
 2   monto            1000 non-null   float64
 3   descripcion      1000 non-null   str    
dtypes: float64(1), str(3)
memory usage: 66.5 KB


In [6]:
# --- 4. TRATAMIENTO Y NORMALIZACIÓN (Data Wrangling) ---

# A. Estandarización de Series Cronológicas:
# Usamos las nuevas columnas estandarizadas de fecha
df_mayor['fecha_contable'] = pd.to_datetime(df_mayor['fecha_contable'])
df_banco['fecha_banco'] = pd.to_datetime(df_banco['fecha_banco'])

# B. El Principio del Espejo Contable (Ajuste de Signos):
# Invertimos los signos del extracto para que la lógica de comparación sea 1:1
df_banco['monto_normalizado'] = df_banco['monto'] * -1

# C. Validación de la Transformación:
print("--- VALIDACIÓN DE TIPOS DE DATOS ---")
print(f"Tipo de Fecha en Mayor: {df_mayor['fecha_contable'].dtype}")
print(f"Tipo de Fecha en Banco: {df_banco['fecha_banco'].dtype}")

print("\n--- MUESTRA DEL EXTRACTO BANCARIO NORMALIZADO ---")
display(df_banco[['ref_comprobante', 'fecha_banco', 'descripcion', 'monto', 'monto_normalizado']].head())

--- VALIDACIÓN DE TIPOS DE DATOS ---
Tipo de Fecha en Mayor: datetime64[us]
Tipo de Fecha en Banco: datetime64[us]

--- MUESTRA DEL EXTRACTO BANCARIO NORMALIZADO ---


,ref_comprobante,fecha_banco,descripcion,monto,monto_normalizado
0,TRX-1001,2026-03-09,BCO_MOV_PROV,79749.55,-79749.55
1,TRX-1002,2026-03-17,BCO_MOV_CLIE,-92790.60,92790.60
2,TRX-1003,2026-03-20,BCO_MOV_AGUA,24229.57,-24229.57
3,TRX-1004,2026-03-11,BCO_MOV_PROV,40532.28,-40532.28
4,TRX-1005,2026-03-17,BCO_MOV_CLIE,-16089.19,16089.19
